# FDM-PINN plate bending under a SYMMETRIC TRIANGULAR ("tent") loading

In [1]:
"""
FDM-PINN (raw (x,y) input, Finite-Difference residual block) for Kirchhoff-Love
plate bending under a SYMMETRIC TRIANGULAR ("tent") loading, all-CLAMPED /
FIXED edges (FFFF case: Fixed-Fixed-Fixed-Fixed, equivalently CCCC in some
notations).

Load definition (unchanged from the SS-SS-SS-SS triangular script):
    q(x) = q0 * (2x/a)         for 0   <= x <= a/2
    q(x) = q0 * (2(a-x)/a)     for a/2 <= x <= a
    (uniform along y, i.e. independent of y)

This is a direct adaptation of the SS-SS-SS-SS triangular FDM-PINN script.
ALL of the FD-stencil fixes documented below (the true-4th-derivative fix,
the 2-ring interior mask, the L-BFGS fine-tuning phase, the grid-resolution
bump) are properties of the finite-difference residual block and the load
being x-dependent -- none of them are simply-supported-specific, so they
are carried over completely unchanged. Only the BOUNDARY CONDITION changes
in this version; see the new section below the original fix log for what
that changes and why.

-----------------------------------------------------------------------------
ORIGINAL FIX LOG (carried over unchanged -- these are FD-stencil / training
fixes, independent of the SS vs. clamped BC choice)
-----------------------------------------------------------------------------
1) BIHARMONIC STENCIL BUG (root cause of most of the error in an earlier
   version): w_xxxx / w_yyyy must be TRUE central 5-point 4th-derivative
   stencils applied directly to F, never composed from two 2nd-derivative
   passes in the same direction (composing amplifies one-sided boundary
   error by ~1/h^2). The PDE residual is masked to the 2-ring interior
   ([2:-2, 2:-2]) so it's never evaluated where the 4th-derivative stencil
   would need to reach past the domain edge. w_xxyy (mixed term) is safely
   built by composing 2nd-derivative passes in DIFFERENT directions.

2) LOSS IMBALANCE (SS case only, see below): the SS-SS-SS-SS version
   needed a MOMENT_LOSS_WEIGHT to keep the soft Mx=My=0 edge loss from
   dominating pde_loss. THIS DOES NOT APPLY HERE -- see the new section
   below; the clamped case has no soft moment term at all.

3) DIAGNOSTIC: epochs_to_early_stop was hitting MAX_EPOCHS in an earlier
   version, motivating the L-BFGS fine-tuning phase (fix 5 below).

4) FD TRUNCATION FLOOR: a 2nd-order-accurate FD biharmonic operator has a
   real O(h^2) discretization error that autograd-based PINNs don't have.
   Grid resolution (GRID_NX/GRID_NY) is configurable and defaults to 81.

5) OPTIMIZATION FLOOR: a short L-BFGS fine-tuning phase runs after Adam
   plateaus (or hits MAX_EPOCHS), plus a slightly larger network
   (hidden 64->96, layers 3->4).

-----------------------------------------------------------------------------
NEW: CHANGE TO CLAMPED (FFFF) BOUNDARY CONDITION + REFERENCE SOLUTION
-----------------------------------------------------------------------------
Same underlying reasoning as the FFFF uniform-load and FFFF patch-load
scripts (see those docstrings for the full derivation); summarized here:

  1. HARD-BC ENVELOPE: simply-supported edges are w=0, Mn=0 (moment
     natural BC); clamped edges are w=0, dw/dn=0 (BOTH essential/
     geometric). The envelope is squared, [xi(1-xi)]^2 * [eta(1-eta)]^2,
     so w AND its normal derivative are exactly zero on all four edges by
     construction -- verified: d/dxi[xi^2(1-xi)^2] = 0 at xi=0,1.

  2. LOSS FUNCTION: because BOTH clamped-edge conditions are hard-baked
     into the network output, there is no natural (moment) BC left to
     enforce softly. MOMENT_LOSS_WEIGHT and moment_loss are REMOVED
     entirely (not just set to a different value) -- total training loss
     is pde_loss only. The former moment diagnostics are replaced by a
     slope_bc_loss diagnostic (dw/dn at the boundary, computed with the
     SAME finite-difference derivative helpers already used elsewhere in
     this script -- d1_dx_full / d1_dy_full -- rather than autograd, to
     stay consistent with this being the FDM-PINN variant). Like bc_loss,
     it is a diagnostic only; its value does not feed into `total`.

  3. REFERENCE / "ANALYTICAL" SOLUTION: no Navier series exists for a
     clamped plate under ANY load (the issue is the edge condition, not
     the load shape -- see the FFFF uniform-load script's docstring for
     why sine terms can't satisfy dw/dn=0 term-by-term). The reference
     here is the same verified high-resolution finite-difference clamped-
     biharmonic solver used in the FFFF uniform-load and FFFF patch-load
     scripts, generalized to accept this script's triangular load as its
     forcing term. Unlike the patch load (a step discontinuity, which
     needed a floating-point tolerance fix to converge cleanly -- see the
     FFFF patch script), the triangular load is CONTINUOUS in x (only a
     slope kink at x=a/2, no value discontinuity), so the FD reference
     solver converges smoothly under grid refinement without needing any
     such fix; this was verified directly (center deflection changes by
     <0.3% between 41x41 and 121x121 reference grids, monotonically).
     As with the patch case, there is no single external literature
     coefficient for this load shape to compare against -- grid-refinement
     convergence is the validation used here.

NOTE: the FD reference solver's own grid (_FD_REF_RES) is independent of,
and by default finer than, GRID_NX/GRID_NY (the PINN's own FD-residual
training grid) -- the two serve different purposes and don't need to match.
-----------------------------------------------------------------------------

Outputs (all written to ./results/<MODEL_NAME>_<LOADING_CASE>/):
  - best_weights.pt, training_convergence.csv, cost_metrics.json,
    error_metrics.json, loss_balance.json, field_data.csv, figures/*.png

Run:
    python fdm_pinn_triangular_hardBC_FFFF.py
"""

import os
import json
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import cm

from scipy.sparse import csr_matrix
from scipy.sparse.linalg import spsolve
from scipy.interpolate import RegularGridInterpolator

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "FDM_PINN_HardBC_FFFF"  # >>> VARIANT POINT: change per architecture
LOADING_CASE = "triangular"          # symmetric tent load, peak at x=a/2
BC_CASE = "FFFF"                     # Fixed (clamped) on all four edges: w=0 and dw/dn=0

OUT_DIR = os.path.join("results", f"{MODEL_NAME}_{LOADING_CASE}")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
MAX_EPOCHS = 15000
PATIENCE = 3000
PRINT_EVERY = 100
LR_INIT = 1e-3
LR_DECAY_STEPS = 2000
LR_DECAY_RATE = 0.95

# --- grid resolution for the PINN's own FD-residual training grid (carried
# over from the SS-SS-SS-SS version's fix (4); see original fix log above).
GRID_NX = 81
GRID_NY = 81

# --- short L-BFGS fine-tuning phase (carried over from fix (5) above).
LBFGS_STEPS = 300
LBFGS_LR = 1.0
LBFGS_HISTORY_SIZE = 50

# Resolution of the high-fidelity finite-difference grid used to build the
# clamped-plate reference/benchmark solution (independent of GRID_NX/NY --
# see docstring). Same default as the other FFFF scripts.
_FD_REF_RES = 101

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# ============================================================
# SEEDING
# ============================================================
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================================
# PHYSICAL / NON-DIMENSIONAL PARAMETERS
# ============================================================
real_a, real_b, real_h = 1.0, 1.0, 0.01
real_E, real_nu, real_q0 = 200e9, 0.3, 1000.0

D_dim = real_E * (real_h ** 3) / (12 * (1 - real_nu ** 2))
L = real_a
W = real_q0 * (L ** 4) / D_dim
D_nd = 1.0
a_nd, b_nd, q0_nd, nu_nd = 1.0, 1.0, 1.0, real_nu

# ============================================================
# DOMAIN
# ============================================================
nx, ny = GRID_NX, GRID_NY
x_nd = np.linspace(0, a_nd, nx)
y_nd = np.linspace(0, b_nd, ny)
X_nd, Y_nd = np.meshgrid(x_nd, y_nd)   # 'xy' indexing -> shape (ny, nx)
grid_points_nd = np.vstack([X_nd.ravel(), Y_nd.ravel()]).T

X_grid = torch.tensor(grid_points_nd, dtype=DTYPE, device=DEVICE)

# grid spacing, needed by the FD stencils
DX = a_nd / (nx - 1)
DY = b_nd / (ny - 1)

center_x, center_y = a_nd / 2, b_nd / 2
center_idx = int(np.argmin((grid_points_nd[:, 0] - center_x) ** 2 +
                            (grid_points_nd[:, 1] - center_y) ** 2))

# --- Full boundary -- kept for the diagnostic bc_loss / slope_bc_loss -----
boundary_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 1] == 0)[0],
    np.where(grid_points_nd[:, 1] == b_nd)[0],
    np.where(grid_points_nd[:, 0] == 0)[0],
    np.where(grid_points_nd[:, 0] == a_nd)[0],
]))
bc_idx = torch.tensor(boundary_indices, dtype=torch.long, device=DEVICE)

# ============================================================
# LOAD DEFINITION (numpy version, for the FD reference solver, and torch
# version, for the FDM-PINN training residual)
# ============================================================
def _triangular_load_np(x, y):
    """Numpy version of the symmetric tent load, used to build the FD
    reference solution."""
    ramp_up = 2.0 * x / a_nd
    ramp_down = 2.0 * (a_nd - x) / a_nd
    return q0_nd * np.where(x <= a_nd / 2, ramp_up, ramp_down)


def triangular_load_nd(x, y):
    """Symmetric tent load: 0 at x=0 and x=a, peak q0 at x=a/2, uniform in y.
    x, y are torch tensors (y unused but kept for a consistent call signature)."""
    ramp_up = 2.0 * x / a_nd
    ramp_down = 2.0 * (a_nd - x) / a_nd
    q_x = torch.where(x <= a_nd / 2, ramp_up, ramp_down)
    return q0_nd * q_x


# ============================================================
# REFERENCE SOLUTION (high-resolution finite-difference, CLAMPED all
# edges, triangular load). See module docstring for why this replaces the
# Navier series used in the SS-SS-SS-SS triangular script, and for caveats.
# ============================================================
def _solve_clamped_biharmonic_fd(nx_ref, ny_ref, load_func):
    """
    Finite-difference solution of D_nd * grad^4(w) = q(x,y) on
    [0,a_nd] x [0,b_nd] with CLAMPED (w=0 AND dw/dn=0) boundary conditions
    on all four edges, for an ARBITRARY load field `load_func(X, Y)`.

    Identical 13-point biharmonic stencil and single-layer clamped
    ghost-node mirroring as the other FFFF scripts (see the FFFF uniform-
    load script's docstring for the derivation and independent stencil
    verification); only the right-hand side varies per grid point here.

    Returns (x_ref, y_ref, w_ref, Mx_ref, My_ref, Mxy_ref): 1D coordinate
    arrays and 2D fields of shape (ny_ref, nx_ref).
    """
    Nx, Ny = nx_ref, ny_ref
    hx = a_nd / (Nx - 1)
    hy = b_nd / (Ny - 1)

    x_ref = np.linspace(0, a_nd, Nx)
    y_ref = np.linspace(0, b_nd, Ny)
    X_ref_grid, Y_ref_grid = np.meshgrid(x_ref, y_ref)   # shape (Ny, Nx), matches w_ref indexing
    q_ref_grid = load_func(X_ref_grid, Y_ref_grid)

    n_unknown_y = Ny - 2

    def idx(i, j):
        return (i - 1) * n_unknown_y + (j - 1)

    stencil = []
    for di, c in [(-2, 1.0), (-1, -4.0), (0, 6.0), (1, -4.0), (2, 1.0)]:
        stencil.append((di, 0, c / hx ** 4))
    for dj, c in [(-2, 1.0), (-1, -4.0), (0, 6.0), (1, -4.0), (2, 1.0)]:
        stencil.append((0, dj, c / hy ** 4))
    mixed = [(-1, -1, 1.0), (0, -1, -2.0), (1, -1, 1.0),
             (-1, 0, -2.0), (0, 0, 4.0), (1, 0, -2.0),
             (-1, 1, 1.0), (0, 1, -2.0), (1, 1, 1.0)]
    for di, dj, c in mixed:
        stencil.append((di, dj, 2.0 * c / (hx ** 2 * hy ** 2)))

    rows, cols, vals = [], [], []
    rhs = np.zeros(n_unknown_y * (Nx - 2), dtype=float)

    for i in range(1, Nx - 1):
        for j in range(1, Ny - 1):
            row = idx(i, j)
            rhs[row] = q_ref_grid[j, i] / D_nd
            for di, dj, c in stencil:
                ii, jj = i + di, j + dj
                if ii == -1:
                    ii = 1
                elif ii == Nx:
                    ii = Nx - 2
                if jj == -1:
                    jj = 1
                elif jj == Ny:
                    jj = Ny - 2
                if ii == 0 or ii == Nx - 1 or jj == 0 or jj == Ny - 1:
                    continue
                rows.append(row)
                cols.append(idx(ii, jj))
                vals.append(c)

    A = csr_matrix((vals, (rows, cols)),
                    shape=(n_unknown_y * (Nx - 2), n_unknown_y * (Nx - 2)))
    u = spsolve(A, rhs)

    w_ref = np.zeros((Ny, Nx))
    for i in range(1, Nx - 1):
        for j in range(1, Ny - 1):
            w_ref[j, i] = u[idx(i, j)]

    w_x = np.gradient(w_ref, x_ref, axis=1)
    w_xx = np.gradient(w_x, x_ref, axis=1)
    w_y = np.gradient(w_ref, y_ref, axis=0)
    w_yy = np.gradient(w_y, y_ref, axis=0)
    w_xy = np.gradient(w_x, y_ref, axis=0)

    Mx_ref = -(w_xx + nu_nd * w_yy)
    My_ref = -(w_yy + nu_nd * w_xx)
    Mxy_ref = -(1 - nu_nd) * w_xy

    return x_ref, y_ref, w_ref, Mx_ref, My_ref, Mxy_ref


def _check_fd_convergence():
    """Grid-refinement sanity check. Unlike the patch-load case, the
    triangular load is continuous (only a slope kink at x=a/2), so no
    boundary-tolerance fix is needed for clean convergence here."""
    _, _, w_lo, _, _, _ = _solve_clamped_biharmonic_fd(61, 61, _triangular_load_np)
    _, _, w_hi, _, _, _ = _solve_clamped_biharmonic_fd(121, 121, _triangular_load_np)
    c_lo = w_lo[30, 30]
    c_hi = w_hi[60, 60]
    rel_diff = abs(c_lo - c_hi) / (abs(c_hi) + 1e-30)
    print(f"[FD reference check] center-w relative change 61x61 -> 121x121 grid: {rel_diff:.3e}")
    if rel_diff > 1e-3:
        print("  WARNING: FD reference solution may not be fully grid-converged; "
              "consider increasing _FD_REF_RES.")
    print("  (No single external literature coefficient exists for this load "
          "shape, unlike the FFFF uniform-load case's ~0.00126 benchmark -- "
          "grid-refinement convergence is the validation used here.)")


_check_fd_convergence()

print(f"[Clamped-plate reference] solving high-resolution FD biharmonic system "
      f"at {_FD_REF_RES}x{_FD_REF_RES} for the FFFF + {LOADING_CASE} benchmark reference...")
_x_ref, _y_ref, _w_ref, _Mx_ref, _My_ref, _Mxy_ref = _solve_clamped_biharmonic_fd(
    _FD_REF_RES, _FD_REF_RES, _triangular_load_np)

_w_interp = RegularGridInterpolator((_y_ref, _x_ref), _w_ref)
_Mx_interp = RegularGridInterpolator((_y_ref, _x_ref), _Mx_ref)
_My_interp = RegularGridInterpolator((_y_ref, _x_ref), _My_ref)
_Mxy_interp = RegularGridInterpolator((_y_ref, _x_ref), _Mxy_ref)


def _interp_field(interp, x, y):
    pts = np.stack([np.asarray(y).ravel(), np.asarray(x).ravel()], axis=-1)
    return interp(pts).reshape(np.asarray(x).shape)


def analytical_w_nd(x, y):
    """Reference (FD-benchmark, not closed-form) non-dim deflection field.
    Named 'analytical_*' to keep parity with the rest of the pipeline /
    downstream aggregation scripts across load cases -- see docstring for
    why this is an FD reference rather than a true closed form here."""
    return _interp_field(_w_interp, x, y)


def analytical_Mx_nd(x, y):
    return _interp_field(_Mx_interp, x, y)


def analytical_My_nd(x, y):
    return _interp_field(_My_interp, x, y)


def analytical_Mxy_nd(x, y):
    return _interp_field(_Mxy_interp, x, y)


# ============================================================
# FINITE-DIFFERENCE STENCILS (unchanged from the SS-SS-SS-SS version --
# these operate on the PINN's own training grid, GRID_NX x GRID_NY, and
# are independent of the boundary condition choice)
# ============================================================
# All operate on a full (ny, nx) field and return a same-shaped field.
# Plain slicing + arithmetic, so gradients flow straight back through to
# the network weights.

def d2_dx2_full(F, h):
    """2nd derivative in x. Central 3-pt interior, one-sided 4-pt at the
    two boundary columns."""
    d2 = torch.zeros_like(F)
    d2[:, 1:-1] = (F[:, :-2] - 2.0 * F[:, 1:-1] + F[:, 2:]) / h ** 2
    d2[:, 0] = (2.0 * F[:, 0] - 5.0 * F[:, 1] + 4.0 * F[:, 2] - F[:, 3]) / h ** 2
    d2[:, -1] = (2.0 * F[:, -1] - 5.0 * F[:, -2] + 4.0 * F[:, -3] - F[:, -4]) / h ** 2
    return d2


def d2_dy2_full(F, h):
    """2nd derivative in y. Central 3-pt interior, one-sided 4-pt at the
    two boundary rows."""
    d2 = torch.zeros_like(F)
    d2[1:-1, :] = (F[:-2, :] - 2.0 * F[1:-1, :] + F[2:, :]) / h ** 2
    d2[0, :] = (2.0 * F[0, :] - 5.0 * F[1, :] + 4.0 * F[2, :] - F[3, :]) / h ** 2
    d2[-1, :] = (2.0 * F[-1, :] - 5.0 * F[-2, :] + 4.0 * F[-3, :] - F[-4, :]) / h ** 2
    return d2


def d1_dx_full(F, h):
    d1 = torch.zeros_like(F)
    d1[:, 1:-1] = (F[:, 2:] - F[:, :-2]) / (2.0 * h)
    d1[:, 0] = (-3.0 * F[:, 0] + 4.0 * F[:, 1] - F[:, 2]) / (2.0 * h)
    d1[:, -1] = (3.0 * F[:, -1] - 4.0 * F[:, -2] + F[:, -3]) / (2.0 * h)
    return d1


def d1_dy_full(F, h):
    d1 = torch.zeros_like(F)
    d1[1:-1, :] = (F[2:, :] - F[:-2, :]) / (2.0 * h)
    d1[0, :] = (-3.0 * F[0, :] + 4.0 * F[1, :] - F[2, :]) / (2.0 * h)
    d1[-1, :] = (3.0 * F[-1, :] - 4.0 * F[-2, :] + F[-3, :]) / (2.0 * h)
    return d1


# --- TRUE 4th-derivative stencils, computed directly from F (not by
# composing d2_dx2_full/d2_dy2_full with itself -- see original fix log).
def d4_dx4_full(F, h):
    d4 = torch.zeros_like(F)
    d4[:, 2:-2] = (F[:, :-4] - 4.0 * F[:, 1:-3] + 6.0 * F[:, 2:-2]
                   - 4.0 * F[:, 3:-1] + F[:, 4:]) / h ** 4
    # one-sided 4th derivative, 2nd-order accurate (6-pt), for the two
    # columns nearest each edge -- diagnostic fill only, not used in loss
    d4[:, 0] = (3.0 * F[:, 0] - 14.0 * F[:, 1] + 26.0 * F[:, 2]
                - 24.0 * F[:, 3] + 11.0 * F[:, 4] - 2.0 * F[:, 5]) / h ** 4
    d4[:, 1] = d4[:, 0]
    d4[:, -1] = (3.0 * F[:, -1] - 14.0 * F[:, -2] + 26.0 * F[:, -3]
                 - 24.0 * F[:, -4] + 11.0 * F[:, -5] - 2.0 * F[:, -6]) / h ** 4
    d4[:, -2] = d4[:, -1]
    return d4


def d4_dy4_full(F, h):
    d4 = torch.zeros_like(F)
    d4[2:-2, :] = (F[:-4, :] - 4.0 * F[1:-3, :] + 6.0 * F[2:-2, :]
                   - 4.0 * F[3:-1, :] + F[4:, :]) / h ** 4
    d4[0, :] = (3.0 * F[0, :] - 14.0 * F[1, :] + 26.0 * F[2, :]
                - 24.0 * F[3, :] + 11.0 * F[4, :] - 2.0 * F[5, :]) / h ** 4
    d4[1, :] = d4[0, :]
    d4[-1, :] = (3.0 * F[-1, :] - 14.0 * F[-2, :] + 26.0 * F[-3, :]
                 - 24.0 * F[-4, :] + 11.0 * F[-5, :] - 2.0 * F[-6, :]) / h ** 4
    d4[-2, :] = d4[-1, :]
    return d4


def fd_plate_fields(w_flat):
    """From the flat network output w (N,1), returns the (ny,nx) field and
    the derivative fields needed for the PDE residual / moments / twisting
    moment / clamped-slope diagnostic, all via finite differences (no
    autograd.grad).

    w_x, w_y: central 3-pt interior / one-sided 4-pt boundary. Used here
              for the clamped slope diagnostic (dw/dn at the boundary),
              in addition to Mxy as in the SS-SS-SS-SS version.
    w_xx, w_yy: same stencil family, used for the moment fields (still
                computed and plotted even though they no longer feed a
                soft loss term -- they're physically meaningful outputs).
    w_xxxx, w_yyyy: TRUE central 5-pt 4th derivative computed directly
                on F (see original fix log -- no double-composition).
    w_xxyy: mixed term, composed from a 2nd-derivative pass in y on top of
            a 2nd-derivative pass in x (different directions, so it never
            re-differentiates a one-sided value in the same axis).
    """
    F = w_flat.reshape(ny, nx)
    w_xx = d2_dx2_full(F, DX)
    w_yy = d2_dy2_full(F, DY)
    w_xxxx = d4_dx4_full(F, DX)
    w_yyyy = d4_dy4_full(F, DY)
    w_xxyy = d2_dy2_full(w_xx, DY)
    w_x = d1_dx_full(F, DX)
    w_y = d1_dy_full(F, DY)
    w_xy = d1_dy_full(w_x, DY)
    return F, w_xx, w_yy, w_xxxx, w_yyyy, w_xxyy, w_x, w_y, w_xy


# ============================================================
# MODEL   (>>> VARIANT POINT: swap the input block for FF-PINN / FF+FDM-PINN)
# ============================================================
class PlatePINN(nn.Module):
    def __init__(self, hidden=96, n_layers=4):
        super().__init__()
        layers = []
        in_dim = 2
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden, dtype=DTYPE))
            layers.append(nn.SiLU())
            in_dim = hidden
        self.hidden = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, 1, dtype=DTYPE)
        self.scale = nn.Parameter(torch.tensor(1.0, dtype=DTYPE))

    def forward(self, x, y):
        h = torch.cat([x / a_nd, y / b_nd], dim=1)   # >>> VARIANT POINT: Fourier embed here for FF variants
        h = self.hidden(h)
        raw = self.out(h)

        # Hard-constrained CLAMPED BC: w=0 AND dw/dn=0 on all four edges.
        # Squaring each factor (vs. the SS case's unsquared xi(1-xi)) makes
        # the envelope's normal derivative also vanish at each edge:
        #   d/dxi[xi^2(1-xi)^2] = 2xi(1-xi)^2 - 2xi^2(1-xi) = 0 at xi=0,1
        # so both w and dw/dxi (and likewise dw/deta) are identically zero
        # on the boundary by construction, for any smooth 'raw' output.
        xi = x / a_nd
        eta = y / b_nd
        envelope = (xi * (1 - xi)) ** 2 * (eta * (1 - eta)) ** 2
        return self.scale * envelope * raw


model = PlatePINN().to(DEVICE)

# ============================================================
# OPTIMIZER + LR SCHEDULE
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: LR_DECAY_RATE ** (step / LR_DECAY_STEPS)
)


# ============================================================
# LOSS   (>>> VARIANT POINT: RESIDUAL BLOCK -- finite differences, not autograd)
# ============================================================
def compute_loss():
    x = X_grid[:, 0:1]
    y = X_grid[:, 1:2]

    w = model(x, y)   # flat (N,1); no requires_grad_ needed, we never call autograd.grad
    F, w_xx, w_yy, w_xxxx, w_yyyy, w_xxyy, w_x, w_y, _ = fd_plate_fields(w)

    q_flat = triangular_load_nd(x, y)
    q_full = q_flat.reshape(ny, nx)

    biharmonic = w_xxxx + 2.0 * w_xxyy + w_yyyy
    pde_residual = biharmonic - q_full
    # 2-ring interior mask (see original fix log): the true central
    # 4th-derivative stencil is only valid 2 points in from each edge.
    pde_loss = torch.mean(pde_residual[2:-2, 2:-2] ** 2)

    # DIAGNOSTICS ONLY -- both should be ~0 exactly, by construction of the
    # squared hard-BC envelope. Clamped edges have no natural (moment) BC
    # left to enforce, so unlike the SS-SS-SS-SS triangular script there is
    # NO soft boundary loss term feeding into `total` below.
    bc_loss = torch.mean(w[bc_idx] ** 2)
    slope_bc_loss = (torch.mean(w_x[:, 0] ** 2) + torch.mean(w_x[:, -1] ** 2) +
                      torch.mean(w_y[0, :] ** 2) + torch.mean(w_y[-1, :] ** 2))

    total = pde_loss
    return total, pde_loss, bc_loss, slope_bc_loss, w


# ============================================================
# TRAINING LOOP with computational-cost logging + early stopping
# ============================================================
epoch_history, loss_history, pde_history, bc_history, slope_bc_history = [], [], [], [], []
w_center_history, epoch_time_history = [], []

best_loss = float("inf")
wait = 0
weights_path = os.path.join(OUT_DIR, "best_weights.pt")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

train_start = time.perf_counter()
converged_epoch = MAX_EPOCHS

for epoch in range(MAX_EPOCHS):
    epoch_t0 = time.perf_counter()

    optimizer.zero_grad()
    total, pde, bc, slope_bc, w_pred = compute_loss()
    total.backward()
    optimizer.step()
    scheduler.step()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    epoch_dt = time.perf_counter() - epoch_t0

    loss_val = total.item()
    w_center_val = w_pred[center_idx, 0].item()

    epoch_history.append(epoch)
    loss_history.append(loss_val)
    pde_history.append(pde.item())
    bc_history.append(bc.item())
    slope_bc_history.append(slope_bc.item())
    w_center_history.append(w_center_val)
    epoch_time_history.append(epoch_dt)

    if epoch % PRINT_EVERY == 0:
        print(f"Epoch {epoch} Loss {loss_val:.6e} (pde {pde.item():.3e}) "
              f"hardBC_w_residual {bc.item():.3e} hardBC_slope_residual {slope_bc.item():.3e} "
              f"Center {w_center_val:.6e} ({epoch_dt*1000:.2f} ms/epoch)")

    if loss_val < best_loss:
        best_loss = loss_val
        wait = 0
        torch.save(model.state_dict(), weights_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}")
            converged_epoch = epoch
            break

# --- L-BFGS fine-tuning phase (carried over unchanged; see original fix
# log item 5) -----------------------------------------------------------
lbfgs_history = []
if LBFGS_STEPS > 0:
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    lbfgs_opt = torch.optim.LBFGS(
        model.parameters(), lr=LBFGS_LR, max_iter=LBFGS_STEPS,
        history_size=LBFGS_HISTORY_SIZE, line_search_fn="strong_wolfe",
        tolerance_grad=1e-12, tolerance_change=1e-14,
    )

    def lbfgs_closure():
        lbfgs_opt.zero_grad()
        total_l, pde_l, bc_l, slope_bc_l, w_pred_l = compute_loss()
        total_l.backward()
        lbfgs_history.append((total_l.item(), pde_l.item(), slope_bc_l.item()))
        return total_l

    print(f"Starting L-BFGS fine-tuning ({LBFGS_STEPS} steps)...")
    lbfgs_t0 = time.perf_counter()
    lbfgs_opt.step(lbfgs_closure)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    lbfgs_dt = time.perf_counter() - lbfgs_t0

    with torch.no_grad():
        final_total, final_pde_l, final_bc_l, final_slope_bc_l, _ = compute_loss()
    print(f"L-BFGS finished in {lbfgs_dt:.2f}s | final loss {final_total.item():.6e} "
          f"(pde {final_pde_l.item():.3e})")

    # Only keep the L-BFGS result if it actually improved the loss (LBFGS
    # with strong Wolfe line search can occasionally step to a worse point
    # on a highly nonconvex loss surface -- guard against that).
    if final_total.item() < best_loss:
        best_loss = final_total.item()
        torch.save(model.state_dict(), weights_path)
        print("L-BFGS improved on the best Adam checkpoint -- keeping it.")
    else:
        print("L-BFGS did not improve on the best Adam checkpoint -- keeping Adam weights.")
else:
    lbfgs_dt = 0.0

total_train_time = time.perf_counter() - train_start
peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else float("nan")

model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
model.eval()

# ============================================================
# SAVE TRAINING CONVERGENCE CSV
# ============================================================
w_center_dim_history = np.array(w_center_history) * W
df_hist = pd.DataFrame({
    "epoch": epoch_history,
    "total_loss": loss_history,
    "pde_loss": pde_history,
    "hard_w_bc_residual_diagnostic": bc_history,
    "hard_slope_bc_residual_diagnostic": slope_bc_history,
    "w_center_nd": w_center_history,
    "w_center_dim": w_center_dim_history,
    "epoch_time_s": epoch_time_history,
})
df_hist.to_csv(os.path.join(OUT_DIR, "training_convergence.csv"), index=False)
print("Saved training_convergence.csv")

# ============================================================
# LOSS BALANCE DIAGNOSTIC
# ============================================================
final_pde = pde_history[-1]
final_bc = bc_history[-1]
final_slope_bc = slope_bc_history[-1]
loss_balance = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_case": BC_CASE,
    "final_pde_loss": final_pde,
    "final_hard_w_bc_residual_diagnostic": final_bc,
    "final_hard_slope_bc_residual_diagnostic": final_slope_bc,
    "note": ("Total training loss = pde_loss only. Both essential clamped "
             "BCs (w=0 and dw/dn=0) are hard-enforced via the squared "
             "envelope, so there is no soft/natural boundary loss term to "
             "balance against the PDE term here (unlike the SS-SS-SS-SS "
             "triangular script, which needed MOMENT_LOSS_WEIGHT to "
             "rebalance a soft moment-BC loss)."),
}
if final_bc > 1e-20 or final_slope_bc > 1e-14:
    print(f"WARNING: hard BC residual diagnostics are not ~0 "
          f"(w: {final_bc:.3e}, slope: {final_slope_bc:.3e}) -- double-check "
          f"the envelope function / grid point exactness (float64 expected). "
          f"Note: unlike the pure-autograd FFFF scripts, these diagnostics "
          f"are computed with the same FD stencils as training here, so a "
          f"nonzero-but-tiny value from FD truncation error is more plausible "
          f"than in the autograd-based FFFF scripts.")
with open(os.path.join(OUT_DIR, "loss_balance.json"), "w") as f:
    json.dump(loss_balance, f, indent=2)
print("Saved loss_balance.json:", loss_balance)

# ============================================================
# COMPUTATIONAL COST METRICS
# ============================================================
cost_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_case": BC_CASE,
    "device": str(DEVICE),
    "epochs_run": len(epoch_history),
    "epochs_to_early_stop": converged_epoch,
    "epochs_to_convergence": converged_epoch,  # deprecated alias, same value
    "mean_time_per_epoch_s": float(np.mean(epoch_time_history)),
    "median_time_per_epoch_s": float(np.median(epoch_time_history)),
    "total_training_time_s": total_train_time,
    "lbfgs_steps_requested": LBFGS_STEPS,
    "lbfgs_time_s": lbfgs_dt,
    "peak_gpu_memory_mb": peak_mem_mb,
    "n_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "grid_nx": GRID_NX,
    "grid_ny": GRID_NY,
    "fd_reference_resolution": _FD_REF_RES,
}
with open(os.path.join(OUT_DIR, "cost_metrics.json"), "w") as f:
    json.dump(cost_metrics, f, indent=2)
print("Saved cost_metrics.json:", cost_metrics)

# ============================================================
# POST-PROCESSING: full-field predictions (w, Mx, My, Mxy) via finite
# differences (same helper the loss uses -- no autograd.grad here either).
# ============================================================
def compute_full_fields():
    x = X_grid[:, 0:1]
    y = X_grid[:, 1:2]
    w = model(x, y)
    F, w_xx, w_yy, _, _, _, w_x, w_y, w_xy = fd_plate_fields(w)

    Mx = -(w_xx + nu_nd * w_yy)
    My = -(w_yy + nu_nd * w_xx)
    Mxy = -(1 - nu_nd) * w_xy

    w_grid = F.detach().cpu().numpy()
    Mx_grid = Mx.detach().cpu().numpy()
    My_grid = My.detach().cpu().numpy()
    Mxy_grid = Mxy.detach().cpu().numpy()
    w_x_grid = w_x.detach().cpu().numpy()
    w_y_grid = w_y.detach().cpu().numpy()
    return w_grid, Mx_grid, My_grid, Mxy_grid, w_x_grid, w_y_grid


w_pred_nd, Mx_pred_nd, My_pred_nd, Mxy_pred_nd, w_x_pred_nd, w_y_pred_nd = compute_full_fields()

# "_anal" naming kept for parity with the rest of the multi-load-case
# pipeline; here it means the FD reference/benchmark field (see docstring),
# not a true closed-form analytical solution.
w_anal_nd = analytical_w_nd(X_nd, Y_nd)
Mx_anal_nd = analytical_Mx_nd(X_nd, Y_nd)
My_anal_nd = analytical_My_nd(X_nd, Y_nd)
Mxy_anal_nd = analytical_Mxy_nd(X_nd, Y_nd)

w_dim_pred, w_dim_anal = w_pred_nd * W, w_anal_nd * W
moment_scale = D_dim * W / L ** 2
Mx_dim_pred, Mx_dim_anal = Mx_pred_nd * moment_scale, Mx_anal_nd * moment_scale
My_dim_pred, My_dim_anal = My_pred_nd * moment_scale, My_anal_nd * moment_scale
Mxy_dim_pred, Mxy_dim_anal = Mxy_pred_nd * moment_scale, Mxy_anal_nd * moment_scale

w_signed_error = w_dim_pred - w_dim_anal
w_error = np.abs(w_signed_error)
Mx_error = np.abs(Mx_dim_pred - Mx_dim_anal)
My_error = np.abs(My_dim_pred - My_dim_anal)
Mxy_error = np.abs(Mxy_dim_pred - Mxy_dim_anal)

# ============================================================
# ERROR METRICS
# ============================================================
ix, iy = nx // 2, ny // 2
w_pred_center, w_anal_center = w_dim_pred[iy, ix], w_dim_anal[iy, ix]
center_abs_err = abs(w_pred_center - w_anal_center)
center_pct_err = center_abs_err / abs(w_anal_center) * 100

L2_error_w = np.sqrt(np.mean((w_dim_pred - w_dim_anal) ** 2))
rel_L2_error_w = np.linalg.norm(w_dim_pred - w_dim_anal) / np.linalg.norm(w_dim_anal)

boundary_w_max_abs = float(np.max(np.abs(w_dim_pred[0, :]))
                            + np.max(np.abs(w_dim_pred[-1, :]))
                            + np.max(np.abs(w_dim_pred[:, 0]))
                            + np.max(np.abs(w_dim_pred[:, -1])))
boundary_slope_x_max_abs = float(max(np.max(np.abs(w_x_pred_nd[:, 0])),
                                      np.max(np.abs(w_x_pred_nd[:, -1]))))
boundary_slope_y_max_abs = float(max(np.max(np.abs(w_y_pred_nd[0, :])),
                                      np.max(np.abs(w_y_pred_nd[-1, :]))))

error_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_case": BC_CASE,
    "fd_reference_resolution": _FD_REF_RES,
    "center_deflection_pred_m": float(w_pred_center),
    "center_deflection_anal_m": float(w_anal_center),
    "center_abs_error_m": float(center_abs_err),
    "center_pct_error": float(center_pct_err),
    "w_max_abs_error_m": float(np.max(w_error)),
    "w_rms_error_m": float(np.sqrt(np.mean(w_error ** 2))),
    "w_L2_error_m": float(L2_error_w),
    "w_relative_L2_error": float(rel_L2_error_w),
    "Mx_max_abs_error_Nm": float(np.max(Mx_error)),
    "Mx_rms_error_Nm": float(np.sqrt(np.mean(Mx_error ** 2))),
    "My_max_abs_error_Nm": float(np.max(My_error)),
    "My_rms_error_Nm": float(np.sqrt(np.mean(My_error ** 2))),
    "Mxy_max_abs_error_Nm": float(np.max(Mxy_error)),
    "Mxy_rms_error_Nm": float(np.sqrt(np.mean(Mxy_error ** 2))),
    "boundary_w_max_abs_value_m": boundary_w_max_abs,
    "boundary_slope_x_max_abs_nd": boundary_slope_x_max_abs,
    "boundary_slope_y_max_abs_nd": boundary_slope_y_max_abs,
}
with open(os.path.join(OUT_DIR, "error_metrics.json"), "w") as f:
    json.dump(error_metrics, f, indent=2)
print("Saved error_metrics.json:", error_metrics)

df_field = pd.DataFrame({
    "x_nd": X_nd.ravel(), "y_nd": Y_nd.ravel(),
    "w_dim_anal": w_dim_anal.ravel(), "w_dim_pred": w_dim_pred.ravel(),
    "Mx_dim_anal": Mx_dim_anal.ravel(), "Mx_dim_pred": Mx_dim_pred.ravel(),
    "My_dim_anal": My_dim_anal.ravel(), "My_dim_pred": My_dim_pred.ravel(),
    "Mxy_dim_anal": Mxy_dim_anal.ravel(), "Mxy_dim_pred": Mxy_dim_pred.ravel(),
})
df_field["w_abs_error"] = np.abs(df_field["w_dim_anal"] - df_field["w_dim_pred"])
df_field["w_rel_error"] = df_field["w_abs_error"] / (np.abs(df_field["w_dim_anal"]) + 1e-12)
df_field.to_csv(os.path.join(OUT_DIR, "field_data.csv"), index=False)
print("Saved field_data.csv")

# ============================================================
# PLOTTING
# ============================================================
plt.rcParams.update({"figure.dpi": 150})


def save_contour(field, title, cbar_label, fname, cmap=cm.jet):
    plt.figure(figsize=(7, 6))
    cf = plt.contourf(X_nd, Y_nd, field, levels=50, cmap=cmap)
    plt.colorbar(cf, label=cbar_label)
    plt.title(title, fontstyle="italic", fontsize=13)
    plt.xlabel("x/a", fontstyle="italic")
    plt.ylabel("y/b", fontstyle="italic")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, fname))
    plt.close()


plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], df_hist["w_center_dim"], linewidth=2, label=f"{MODEL_NAME} Prediction")
plt.axhline(y=w_anal_center, linestyle="--", linewidth=2, color="k", label="FD Reference (Clamped)")
plt.xlabel("Epoch"); plt.ylabel("Center Deflection w (m)")
plt.title("Convergence of Center Deflection")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Center_Deflection.png")); plt.close()

plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], np.abs(df_hist["w_center_dim"] - w_anal_center), linewidth=2)
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Absolute Error at Center (m)")
plt.title("Convergence Error (Log Scale)")
plt.grid(True, which="both", linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Error_Log_Scale.png")); plt.close()

plt.figure(figsize=(8, 6))
plt.plot(df_hist["epoch"], df_hist["total_loss"], label="Total Loss (= PDE Loss)")
plt.plot(df_hist["epoch"], df_hist["hard_w_bc_residual_diagnostic"],
         label="Hard w-BC residual (diagnostic, ~0)", alpha=0.5, linestyle=":")
plt.plot(df_hist["epoch"], df_hist["hard_slope_bc_residual_diagnostic"],
         label="Hard slope-BC residual (diagnostic, ~0)", alpha=0.5, linestyle=":")
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training Loss Curve")
plt.legend(); plt.grid(True, which="both", ls="--", alpha=0.7); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Training_Loss_Curve.png")); plt.close()

save_contour(w_dim_anal, "FD Reference Plate Deflection (Clamped, FFFF)", "Deflection (m)",
             "Analytical_Plate_Deflection.png")
save_contour(w_dim_pred, f"{MODEL_NAME} Predicted Plate Deflection", "Deflection (m)",
             "PINN_Predicted_Plate_Deflection.png")
save_contour(w_error, "Absolute Error in Plate Deflection", "|w_pred - w_ref| (m)",
             "Absolute_Error_in_Plate_Deflection.png", cmap=cm.inferno)
save_contour(w_signed_error, "Signed Error w_pred - w_ref", "w_pred - w_ref (m)",
             "Absolute_Error_w_pred_-_w_anal.png", cmap="coolwarm")

save_contour(Mx_dim_anal, "FD Reference Bending Moment $M_x$ (Clamped)", "Mx (N·m/m)",
             "Bending_Moment_M_x_Analytical.png")
save_contour(Mx_dim_pred, "Predicted Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x.png")
save_contour(Mx_error, "Absolute Error in $M_x$", "|Mx_pred - Mx_ref| (N·m/m)",
             "Absolute_Error_in_M_x.png", cmap="hot")

save_contour(My_dim_anal, "FD Reference Bending Moment $M_y$ (Clamped)", "My (N·m/m)",
             "Bending_Moment_M_y_Analytical.png")
save_contour(My_dim_pred, "Predicted Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y.png")
save_contour(My_error, "Absolute Error in $M_y$", "|My_pred - My_ref| (N·m/m)",
             "Absolute_Error_in_M_y.png", cmap="hot")

save_contour(Mxy_dim_anal, "FD Reference Twisting Moment $M_{xy}$ (Clamped)", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy_Analytical.png")
save_contour(Mxy_dim_pred, "Predicted Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy.png")
save_contour(Mxy_error, "Absolute Error in $M_{xy}$", "|Mxy_pred - Mxy_ref| (N·m/m)",
             "Absolute_Error_in_M_xy.png", cmap="hot")

plt.figure(figsize=(6, 6))
plt.scatter(w_dim_anal.ravel(), w_dim_pred.ravel(), s=8, alpha=0.5)
lims = [min(w_dim_anal.min(), w_dim_pred.min()), max(w_dim_anal.max(), w_dim_pred.max())]
plt.plot(lims, lims, "k--", linewidth=1, label="Perfect agreement")
plt.xlabel("FD Reference displacement w (m)", fontstyle="italic")
plt.ylabel("Predicted displacement w (m)", fontstyle="italic")
plt.title("FD Reference vs Predicted Displacement", fontstyle="italic")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Analytical_vs_Predicted_Displacement.png")); plt.close()

err_fft = np.fft.fftshift(np.fft.fft2(w_error))
err_fft_mag = np.log1p(np.abs(err_fft))
plt.figure(figsize=(7, 6))
plt.imshow(err_fft_mag, cmap="magma", extent=[-nx // 2, nx // 2, -ny // 2, ny // 2])
plt.colorbar(label="log(1 + |FFT error|)")
plt.title(f"Error Spectrum (FFT) — {LOADING_CASE}, {BC_CASE}")
plt.xlabel("kx"); plt.ylabel("ky")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Error_Spectrum_FFT.png")); plt.close()

fig, axes = plt.subplots(4, 3, figsize=(16, 18))
rows = [
    ("Displacement w (m)", w_dim_anal, w_dim_pred, w_error),
    ("Moment Mx (N·m)", Mx_dim_anal, Mx_dim_pred, Mx_error),
    ("Moment My (N·m)", My_dim_anal, My_dim_pred, My_error),
    ("Moment Mxy (N·m)", Mxy_dim_anal, Mxy_dim_pred, Mxy_error),
]
for r, (label, anal, pred, err) in enumerate(rows):
    for c, (title, field, cmap) in enumerate([
        (f"FD Reference {label}", anal, cm.jet),
        (f"Predicted {label}", pred, cm.jet),
        (f"Absolute Error {label}", err, cm.inferno),
    ]):
        ax = axes[r, c]
        cf = ax.contourf(X_nd, Y_nd, field, levels=20, cmap=cmap)
        fig.colorbar(cf, ax=ax)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("x/a"); ax.set_ylabel("y/b")
plt.suptitle(f"{MODEL_NAME} — {LOADING_CASE} loading, {BC_CASE} (clamped, summary grid)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "_summary_grid_all_fields.png")); plt.close()

print(f"\nAll outputs written to: {os.path.abspath(OUT_DIR)}")
print(f"Total training time: {total_train_time:.2f}s | "
      f"Epochs to early-stop: {converged_epoch} | "
      f"Peak GPU mem: {peak_mem_mb:.1f} MB" if DEVICE.type == "cuda" else
      f"\nAll outputs written to: {os.path.abspath(OUT_DIR)} | "
      f"Total training time: {total_train_time:.2f}s | Epochs: {converged_epoch}")

[FD reference check] center-w relative change 61x61 -> 121x121 grid: 1.948e-03
  (No single external literature coefficient exists for this load shape, unlike the FFFF uniform-load case's ~0.00126 benchmark -- grid-refinement convergence is the validation used here.)
[Clamped-plate reference] solving high-resolution FD biharmonic system at 101x101 for the FFFF + triangular benchmark reference...
Epoch 0 Loss 5.714891e-01 (pde 5.715e-01) hardBC_w_residual 0.000e+00 hardBC_slope_residual 1.813e-11 Center -3.418984e-04 (317.51 ms/epoch)
Epoch 100 Loss 3.513474e-02 (pde 3.513e-02) hardBC_w_residual 0.000e+00 hardBC_slope_residual 2.397e-10 Center 7.577643e-04 (27.02 ms/epoch)
Epoch 200 Loss 3.499669e-02 (pde 3.500e-02) hardBC_w_residual 0.000e+00 hardBC_slope_residual 2.358e-10 Center 7.561886e-04 (27.43 ms/epoch)
Epoch 300 Loss 3.493521e-02 (pde 3.494e-02) hardBC_w_residual 0.000e+00 hardBC_slope_residual 2.359e-10 Center 7.564587e-04 (35.47 ms/epoch)
Epoch 400 Loss 3.487645e-02 (pde 3.48

C:\Users\vivek\AppData\Local\Temp\ipykernel_15756\4010431131.py:613: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_locatio

Starting L-BFGS fine-tuning (300 steps)...
L-BFGS finished in 36.07s | final loss 1.870910e-05 (pde 1.871e-05)
L-BFGS improved on the best Adam checkpoint -- keeping it.


C:\Users\vivek\AppData\Local\Temp\ipykernel_15756\4010431131.py:654: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_locatio

Saved training_convergence.csv
Saved loss_balance.json: {'model': 'FDM_PINN_HardBC_FFFF', 'loading_case': 'triangular', 'bc_case': 'FFFF', 'final_pde_loss': 8.193327855069628e-05, 'final_hard_w_bc_residual_diagnostic': 0.0, 'final_hard_slope_bc_residual_diagnostic': 2.982963969767619e-10, 'note': 'Total training loss = pde_loss only. Both essential clamped BCs (w=0 and dw/dn=0) are hard-enforced via the squared envelope, so there is no soft/natural boundary loss term to balance against the PDE term here (unlike the SS-SS-SS-SS triangular script, which needed MOMENT_LOSS_WEIGHT to rebalance a soft moment-BC loss).'}
Saved cost_metrics.json: {'model': 'FDM_PINN_HardBC_FFFF', 'loading_case': 'triangular', 'bc_case': 'FFFF', 'device': 'cuda', 'epochs_run': 15000, 'epochs_to_early_stop': 15000, 'epochs_to_convergence': 15000, 'mean_time_per_epoch_s': 0.031042484513445136, 'median_time_per_epoch_s': 0.030578200006857514, 'total_training_time_s': 516.7734743999317, 'lbfgs_steps_requested': 30